# UFCE3R-30-3 - Big data analytics
# Question 2.4: Data Analysis via PySpark (30 marks)

For this part of the assessment, you will need PySpark. We strongly recommend to use UWE PCs OR Google Colab, as we have already covered this topic in both environments.


## Important Notice 1
After finishing this assessment, you need to submit 2 files:

1) This Jupyter Notebook with your solutions / codes / answers.
2) PDF version of this Jupyter Notebook showing your solutions / codes / answers (convert your final file into PDF).

## Important Notice 2
Please DO NOT change the cells in this notebook. Only write your codes in the specified places. Changing the layout of the notebook/cells, might affect your mark.


# About the Dataset

This dataset presents the raw data about loan applications that were rejected by bank. You can see these variables in the dataset:

**Amount Requested**: The amount of loan requested by customer, in USD.\
**Application Date**: The date customer submitted the application.\
**Loan Title**: The title of the loan.\
**Risk_Score**: The risk score that bank has calculated for the respective loan application.\
**Debt-To-Income Ratio**: The ratio of the debt to income for customer.\
**Zip Code**: Zip code of the customer.\
**State**: The state where customer lives. For example, CO stands for Colorado (USA).\
**Employment Length**: The length of time when the customer has been employed.\
**Policy Code**: You do not need this variable. Throughout this notebook, you will be asked to drop this column.

The dataset `rejected_loan_data.csv` should be in the same directory as this notebook. Or if working with Colab, in the `Files` panel.


## PySpark Setup

Please ensure you have PySpark installed and configured. Use the following command (if needed) to install PySpark:

In [1]:
pip install pyspark findspark

### For Google Colab
If you are using Colab, you MIGHT need to install/upgrade Java. You can use this code:

In [2]:
!pip install py4j
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
print("Java version:")
!java -version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,586 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,289 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [69.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,411 kB]
Ge

Also, you need to upload the dataset (`rejected_loan_data.csv`) to `Files` panel. As this is a large file, give enough time for the system to upload the file.

## Initialise PySpark application
Run this cell to start an application named `Big Data Analytics - PySpark Assessment` for your assessment.

In [3]:
from pyspark.sql import SparkSession, functions as F

# Create or get existing Spark session
try:
    spark
except NameError:
    spark = SparkSession.builder.appName("Big Data Analytics - PySpark Assessment").getOrCreate()


## Q1: Load the CSV into a PySpark DataFrame

Read `rejected_loan_data.csv` with header and infer schema. Save it in `df` and display the first 5 rows.

In [5]:
# BEGIN SOLUTION

from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("RejectedLoanAnalysis").getOrCreate()

# Load CSV with header and inferred schema
df = spark.read.csv(
    "rejected_loan_data.csv",
    header=True,
    inferSchema=True
)

# Display first 5 rows
df.show(5)

# END SOLUTION


+----------------+----------------+--------------------+----------+--------------------+--------+-----+-----------------+-----------+
|Amount Requested|Application Date|          Loan Title|Risk_Score|Debt-To-Income Ratio|Zip Code|State|Employment Length|Policy Code|
+----------------+----------------+--------------------+----------+--------------------+--------+-----+-----------------+-----------+
|          1000.0|      2007-05-26|Wedding Covered b...|     693.0|                 10%|   481xx|   NM|          4 years|        0.0|
|          1000.0|      2007-05-26|  Consolidating Debt|     703.0|                 10%|   010xx|   MA|         < 1 year|        0.0|
|         11000.0|      2007-05-27|Want to consolida...|     715.0|                 10%|   212xx|   MD|           1 year|        0.0|
|          6000.0|      2007-05-27|             waksman|     698.0|              38.64%|   017xx|   MA|         < 1 year|        0.0|
|          1500.0|      2007-05-27|              mdrigo|     5

## Q2: Print Schema, Row Count and Missing Values

Display the schema and the number of rows in the DataFrame. Then find the number of rows missing a value in `Risk_Score`.

In [7]:
# BEGIN SOLUTION

from pyspark.sql.functions import col

# Print schema
df.printSchema()

# Print total number of rows
print("Total number of rows:", df.count())

# Count rows with missing Risk_Score
missing_risk_score = df.filter(col("Risk_Score").isNull()).count()
print("Rows missing Risk_Score:", missing_risk_score)

# END SOLUTION



root
 |-- Amount Requested: double (nullable = true)
 |-- Application Date: date (nullable = true)
 |-- Loan Title: string (nullable = true)
 |-- Risk_Score: string (nullable = true)
 |-- Debt-To-Income Ratio: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Employment Length: string (nullable = true)
 |-- Policy Code: string (nullable = true)

Total number of rows: 10039097
Rows missing Risk_Score: 5688324


## Q3: Filter Loans Above $10,000

Filter all rows where `Amount Requested > 10000`. Save the result to `filtered_df`. Now print the number of rows in `filtered_df`.

In [8]:
# BEGIN SOLUTION

# Filter rows where Amount Requested > 10000
filtered_df = df.filter(col("Amount Requested") > 10000)

# Print number of rows in filtered dataframe
print("Number of loans above $10,000:", filtered_df.count())

# END SOLUTION


Number of loans above $10,000: 4087105


## Q4: Count Loans per State

For the whole dataset (`df`), group by `State` and count the number of loans (rows in the dataset). Save the result to `state_counts`.

In [9]:
# BEGIN SOLUTION

# Group by State and count number of loans
state_counts = df.groupBy("State").count()

# Display results
state_counts.show()

# END SOLUTION


+-----+-------+
|State|  count|
+-----+-------+
|   AZ| 211108|
|   SC| 165870|
|   LA| 153195|
|   MN| 130785|
|   NJ| 318881|
|   DC|  19961|
|   OR| 102599|
|   VA| 274078|
| NULL|     22|
|   RI|  41116|
|   KY| 128642|
|   WY|  18467|
|   NH|  43399|
|   MI| 274849|
|   NV| 120544|
|   WI| 133119|
|   ID|  23801|
|   CA|1196104|
|   CT| 128164|
|   NE|  31880|
+-----+-------+
only showing top 20 rows


## Q5: Dealing with Null Values

From the previous question's results, you can see that there are 22 states missing their value. What do you recommend to do to handle these missing values? Why?

In [11]:
# BEGIN SOLUTION

"""
I recommend replacing missing State values with a category such as 'Unknown'
rather than dropping the rows. This is because the dataset is large and removing
records may lead to unnecessary data loss and biased results. By using an
'Unknown' category, the data remains complete while still allowing state-based
analysis without misrepresenting existing states.
"""

# END SOLUTION


"\nI recommend replacing missing State values with a category such as 'Unknown'\nrather than dropping the rows. This is because the dataset is large and removing\nrecords may lead to unnecessary data loss and biased results. By using an\n'Unknown' category, the data remains complete while still allowing state-based\nanalysis without misrepresenting existing states.\n"

## Q6: Average Amount Requested by Employment Length

Compute the mean of the Amount Requested (`Average Amount Requested`) grouped by `Employment Length`. Save the result to `avg_amount`.

In [12]:
# BEGIN SOLUTION

from pyspark.sql.functions import avg

# Calculate average Amount Requested by Employment Length
avg_amount = (
    df.groupBy("Employment Length")
      .agg(avg("Amount Requested").alias("Average_Amount_Requested"))
)

# Display results
avg_amount.show()

# END SOLUTION


+--------------------+------------------------+
|   Employment Length|Average_Amount_Requested|
+--------------------+------------------------+
|             9 years|      13738.899237590942|
|             5 years|      11356.253200231067|
|                NULL|      11772.703857070997|
|              1 year|      11694.533947989663|
|             2 years|      12973.446237688368|
|             7 years|      15394.898783166595|
|             8 years|      14943.992227494577|
|             4 years|      13625.668976777795|
|             6 years|      14686.090544518458|
|             3 years|      13401.397472643432|
| I could safely g...|                 12450.0|
|           10+ years|      15840.189818435323|
|            < 1 year|       13417.09995168346|
+--------------------+------------------------+



## Q7: Total Requested Amount per State

Compute total `Amount Requested` grouped by `State`. Save the result to `total_amount`.

In [13]:
# BEGIN SOLUTION

from pyspark.sql.functions import sum

# Calculate total Amount Requested by State
total_amount = (
    df.groupBy("State")
      .agg(sum("Amount Requested").alias("Total_Amount_Requested"))
)

# Display results
total_amount.show()

# END SOLUTION


+-----+----------------------+
|State|Total_Amount_Requested|
+-----+----------------------+
|   AZ|  2.7720402084700003E9|
|   SC|       2.05295705762E9|
|   LA|       1.90075725281E9|
|   MN|         1.758631523E9|
|   NJ|       4.53956324876E9|
|   DC|        2.4343228485E8|
|   OR|       1.37791250417E9|
|   VA|       3.58610704994E9|
| NULL|              190500.0|
|   RI|          5.10770169E8|
|   KY|         1.597394169E9|
|   WY|          2.68432475E8|
|   NH|        6.1365108995E8|
|   MI|       3.52463993376E9|
|   NV|       1.57187581795E9|
|   WI|  1.6739458260900002E9|
|   ID|          3.20768849E8|
|   CA|     1.665540410565E10|
|   CT|         1.754707253E9|
|   NE|          4.20026743E8|
+-----+----------------------+
only showing top 20 rows


## Q8: Top 3 Loan Titles by Average Amount

Find top 3 `Loan Title` by average `Amount Requested`. Save the result to `top_titles`.

In [14]:
# BEGIN SOLUTION

# Calculate average loan amount by Loan Title
top_titles = (
    df.groupBy("Loan Title")
      .agg(avg("Amount Requested").alias("Average_Amount"))
      .orderBy(col("Average_Amount").desc())
      .limit(3)
)

# Display results
top_titles.show()

# END SOLUTION


+--------------------+----------------+
|          Loan Title|  Average_Amount|
+--------------------+----------------+
|       Business Loan|66293.1268189762|
|Education and per...|         40000.0|
|           Making it|         35000.0|
+--------------------+----------------+



## Q9: Loans in Colorado with Specific Amount

Find all rows tha corresponds to Colorado (CO) and have the loan amount of more than 15000. Save the result to `filtered_co_loans`. Only show (print) 20 of the rows. Then print the number of rows in `filtered_co_loans`.

In [15]:
# BEGIN SOLUTION

# Filter loans in Colorado with amount > 15000
filtered_co_loans = df.filter(
    (col("State") == "CO") &
    (col("Amount Requested") > 15000)
)

# Show only 20 rows
filtered_co_loans.show(20)

# Print number of matching rows
print("Number of CO loans above $15,000:", filtered_co_loans.count())

# END SOLUTION


+----------------+----------------+--------------------+----------+--------------------+--------+-----+-----------------+-----------+
|Amount Requested|Application Date|          Loan Title|Risk_Score|Debt-To-Income Ratio|Zip Code|State|Employment Length|Policy Code|
+----------------+----------------+--------------------+----------+--------------------+--------+-----+-----------------+-----------+
|         25000.0|      2007-06-26|           San_Diego|     646.0|               7.94%|   921xx|   CO|           1 year|        0.0|
|         25000.0|      2007-08-04|              janeco|     712.0|               84.9%|   800xx|   CO|        10+ years|        0.0|
|         23000.0|      2007-09-13|  Debt consolidation|     717.0|              19.09%|   801xx|   CO|           1 year|        0.0|
|         20000.0|      2007-09-26|              biodoc|     684.0|              28.44%|   920xx|   CO|         < 1 year|        0.0|
|         16500.0|      2007-09-30|            dengerin|     6

## Q10: Your Reflection

In a paragraph less than 100 words, reflect on how you approached this assignment, what was new to you and how you found the solution.

### BEGIN ANSWER
### Your reflection here:
I approached this assignment by breaking down each task into small, logical steps and applying appropriate PySpark transformations and actions. Working with a large-scale dataset strengthened my understanding of distributed data processing and performance-efficient operations. Using groupBy, aggregation functions, and filters in PySpark was particularly valuable, as it differs from Pandas in execution style. Overall, the assignment improved my confidence in handling big data analytically and translating business questions into scalable data solutions.

### END ANSWER